In [ ]:
"""
Improved HydroSource NetCDF Downloader - Multi-SSP Version
- Downloads multiple SSP scenarios (ssp126, ssp245, ssp370, ssp585)
- Parallel downloads with configurable workers
- Resume capability for interrupted downloads
- File integrity verification
- Progress tracking with tqdm
- Automatic retry with exponential backoff
- Skip already completed files
"""

import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pathlib import Path

# ============================================================================
# CONFIGURATION - Edit these values
# ============================================================================
BASE_URL = "https://hydrosource2.ornl.gov/files/SWA9505V3"

# Available GCMs (run once per model):
#   MPI-ESM1-2-HR, ACCESS-CM2, EC-Earth3, CNRM-ESM2-1,
#   BCC-CSM2-MR, MRI-ESM2-0, NorESM2-MM
#
# Variant IDs by model:
#   CNRM-ESM2-1 → r1i1p1f2 ; all others → r1i1p1f1
#
# SSP scenarios: ssp126, ssp245, ssp370, ssp585
#
# Time periods:
#   Historical: 1980–2019
#   Future:     2020–2059, 2060–2099
#
# Variables: prcp (precipitation), streamflow

MODEL = "NorESM2-MM"
SCENARIOS = ["ssp585"]  # Full set: ["ssp126", "ssp245", "ssp370", "ssp585"]
VARIANT = "r1i1p1f1"   # Use r1i1p1f2 for CNRM-ESM2-1
EXPERIMENT = "DBCCA_Daymet"
VARIABLE = "prcp"

# Year range to download
START_YEAR = 2060
END_YEAR = 2099

# Download settings
MAX_WORKERS = 4          # Parallel downloads (don't overload server, 3-6 recommended)
MAX_RETRIES = 5          # Retries per file
CHUNK_SIZE = 1024 * 1024 # 1MB chunks
TIMEOUT = 120            # Request timeout in seconds
VERIFY_SIZE = True       # Verify file size after download
MIN_FILE_SIZE_MB = 1     # Minimum expected file size (MB) - skip if smaller

# Output base directory
OUTPUT_BASE = ""

# ============================================================================
# Helper Functions
# ============================================================================

def get_file_url(year, scenario):
    """Construct URL for a specific year's file."""
    folder = f"{MODEL}_{scenario}_{VARIANT}_{EXPERIMENT}"
    filename = f"{MODEL}_{scenario}_{VARIANT}_{EXPERIMENT}_VIC4_{VARIABLE}_{year}.nc"
    return f"{BASE_URL}/{folder}/{VARIABLE}/{filename}", filename


def get_save_dir(scenario):
    """Get save directory for a scenario."""
    return os.path.join(OUTPUT_BASE, f"{MODEL}_{scenario}_{VARIANT}_{EXPERIMENT}_{VARIABLE}")


def get_remote_file_size(url, timeout=30):
    """Get the expected file size from server headers."""
    try:
        response = requests.head(url, timeout=timeout, allow_redirects=True)
        if response.status_code == 200:
            return int(response.headers.get('content-length', 0))
    except:
        pass
    return None


def is_file_complete(filepath, expected_size=None, min_size_mb=MIN_FILE_SIZE_MB):
    """Check if file exists and appears complete."""
    if not os.path.exists(filepath):
        return False
    
    actual_size = os.path.getsize(filepath)
    
    # If we know expected size, check it
    if expected_size and actual_size >= expected_size:
        return True
    
    # Otherwise, check minimum size threshold
    if actual_size >= min_size_mb * 1024 * 1024:
        return True
    
    return False


def download_file(year, scenario, pbar=None):
    """
    Download a single file with resume capability and retry logic.
    Returns: (year, scenario, success, message)
    """
    url, filename = get_file_url(year, scenario)
    save_dir = get_save_dir(scenario)
    filepath = os.path.join(save_dir, filename)
    
    # Ensure directory exists
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    # Check if already complete
    expected_size = get_remote_file_size(url) if VERIFY_SIZE else None
    
    if is_file_complete(filepath, expected_size):
        return (year, scenario, True, "Already complete")
    
    for attempt in range(MAX_RETRIES):
        try:
            # Check for partial file
            initial_pos = 0
            if os.path.exists(filepath):
                initial_pos = os.path.getsize(filepath)
            
            # Set up headers for resume
            headers = {'User-Agent': 'Mozilla/5.0 HydroSource-Downloader/1.0'}
            if initial_pos > 0:
                headers['Range'] = f'bytes={initial_pos}-'
            
            # Make request
            response = requests.get(url, headers=headers, stream=True, timeout=TIMEOUT)
            
            # Handle response codes
            if response.status_code == 416:  # Range not satisfiable - file complete
                return (year, scenario, True, "Complete (416)")
            elif response.status_code == 404:
                return (year, scenario, False, "File not found (404)")
            elif response.status_code not in [200, 206]:
                raise requests.exceptions.HTTPError(f"HTTP {response.status_code}")
            
            # Determine write mode
            mode = 'ab' if response.status_code == 206 else 'wb'
            if response.status_code == 200 and initial_pos > 0:
                initial_pos = 0  # Server doesn't support resume
            
            # Get total size
            content_length = int(response.headers.get('content-length', 0))
            total_size = content_length + initial_pos
            
            # Download
            with open(filepath, mode) as f:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
            
            # Verify
            actual_size = os.path.getsize(filepath)
            if total_size > 0 and actual_size < total_size * 0.99:  # Allow 1% tolerance
                raise Exception(f"Incomplete: {actual_size}/{total_size} bytes")
            
            return (year, scenario, True, f"Downloaded ({actual_size/1e6:.1f} MB)")
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = min(2 ** attempt, 30)  # Cap at 30 seconds
                time.sleep(wait_time)
            else:
                return (year, scenario, False, f"Failed after {MAX_RETRIES} attempts: {str(e)[:50]}")
    
    return (year, scenario, False, "Unknown error")


def download_all_parallel(years, scenarios, max_workers=MAX_WORKERS):
    """Download all files with parallel workers and progress bar."""
    
    # Create all download tasks (year, scenario pairs)
    tasks = [(year, scenario) for scenario in scenarios for year in years]
    total_files = len(tasks)
    
    results = {scenario: {'success': [], 'failed': [], 'skipped': []} for scenario in scenarios}
    
    print(f"\nDownloading {total_files} files ({len(years)} years × {len(scenarios)} scenarios)")
    print(f"Using {max_workers} parallel workers...")
    print(f"Output base: {OUTPUT_BASE}/\n")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {
            executor.submit(download_file, year, scenario): (year, scenario) 
            for year, scenario in tasks
        }
        
        # Process with progress bar
        with tqdm(total=total_files, desc="Downloading", unit="file") as pbar:
            for future in as_completed(futures):
                year, scenario = futures[future]
                try:
                    year, scenario, success, message = future.result()
                    
                    if success:
                        if "Already" in message:
                            results[scenario]['skipped'].append(year)
                        else:
                            results[scenario]['success'].append(year)
                    else:
                        results[scenario]['failed'].append((year, message))
                    
                    # Update progress bar description
                    status = "✓" if success else "✗"
                    pbar.set_postfix_str(f"{scenario}/{year}: {status}")
                    
                except Exception as e:
                    results[scenario]['failed'].append((year, str(e)))
                
                pbar.update(1)
    
    return results


def print_summary(results, elapsed_time, scenarios):
    """Print download summary."""
    print("\n" + "=" * 70)
    print("DOWNLOAD SUMMARY")
    print("=" * 70)
    
    total_success = 0
    total_skipped = 0
    total_failed = 0
    all_failed = []
    
    for scenario in scenarios:
        r = results[scenario]
        success = len(r['success'])
        skipped = len(r['skipped'])
        failed = len(r['failed'])
        total = success + skipped + failed
        
        total_success += success
        total_skipped += skipped
        total_failed += failed
        all_failed.extend([(scenario, year, msg) for year, msg in r['failed']])
        
        status = "✓" if failed == 0 else "⚠"
        print(f"\n  {status} {scenario}:")
        print(f"      Downloaded: {success:3d}  |  Skipped: {skipped:3d}  |  Failed: {failed:3d}  |  Total: {total}")
    
    grand_total = total_success + total_skipped + total_failed
    
    print("\n" + "-" * 70)
    print(f"  TOTAL:")
    print(f"      Downloaded: {total_success:3d}  |  Skipped: {total_skipped:3d}  |  Failed: {total_failed:3d}  |  Total: {grand_total}")
    print(f"\n  Time elapsed: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
    
    if all_failed:
        print(f"\n  Failed files ({len(all_failed)} total):")
        for scenario, year, error in all_failed[:15]:
            print(f"    {scenario}/{year}: {error}")
        if len(all_failed) > 15:
            print(f"    ... and {len(all_failed) - 15} more")
    
    print("=" * 70)
    
    return total_failed


def test_connectivity(scenarios, years):
    """Test connectivity for each scenario."""
    print("\nTesting connectivity for each scenario...")
    
    all_ok = True
    for scenario in scenarios:
        test_url, test_file = get_file_url(years[0], scenario)
        try:
            response = requests.head(test_url, timeout=30)
            if response.status_code == 200:
                size_mb = int(response.headers.get('content-length', 0)) / 1e6
                print(f"  ✓ {scenario}: accessible ({size_mb:.1f} MB)")
            else:
                print(f"  ✗ {scenario}: HTTP {response.status_code}")
                all_ok = False
        except Exception as e:
            print(f"  ✗ {scenario}: {e}")
            all_ok = False
    
    return all_ok


# ============================================================================
# Main
# ============================================================================
if __name__ == "__main__":
    # Generate year list
    years = list(range(START_YEAR, END_YEAR + 1))
    total_files = len(years) * len(SCENARIOS)
    
    print("=" * 70)
    print("HydroSource NetCDF Downloader - Multi-SSP Version")
    print("=" * 70)
    print(f"  Model:      {MODEL}")
    print(f"  Scenarios:  {', '.join(SCENARIOS)}")
    print(f"  Variable:   {VARIABLE}")
    print(f"  Years:      {START_YEAR} - {END_YEAR} ({len(years)} years)")
    print(f"  Total:      {total_files} files")
    print(f"  Workers:    {MAX_WORKERS}")
    print(f"  Output:     {OUTPUT_BASE}/")
    print("=" * 70)
    
    # Test connectivity
    if not test_connectivity(SCENARIOS, years):
        print("\n⚠ Some scenarios are not accessible. Continue anyway? (y/n): ", end="")
        if input().lower() != 'y':
            print("Aborted.")
            exit(1)
    
    # Create base output directory
    Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)
    
    # Start download
    print("\nStarting downloads...")
    start_time = time.time()
    results = download_all_parallel(years, SCENARIOS, MAX_WORKERS)
    elapsed = time.time() - start_time
    
    # Print summary
    failed_count = print_summary(results, elapsed, SCENARIOS)
    
    # Exit code based on results
    if failed_count > 0:
        print("\nSome files failed. Re-run the script to retry.")
    else:
        print("\n✓ All files downloaded successfully!")